In [1]:
import numpy as np
import plotly.express as px

from extract_visual_features import get_features_objectwise

# dataset is dictionary with keys as object names and values as list of features
# feature vector is 1024-dim vector
dataset = get_features_objectwise(".", use_single_take=True, downsample=5)

In [2]:
len(dataset)

2

In [3]:
f, ef = dataset

In [6]:
ef["bowl"].shape, f["bowl"].shape


(torch.Size([1914, 256]), torch.Size([1914, 1024]))

In [4]:
f.keys()

dict_keys(['bowl', 'knife', 'screwdriver', 'cuttingboard', 'whisk', 'hammer', 'bottle', 'cup', 'banana', 'cereals', 'sponge', 'woodenwedge', 'saw', 'harddrive', 'LeftHand', 'RightHand'])

In [9]:
total_samples = 0
for key in dataset:
    total_samples += dataset[key].shape[0]
    
total_samples/len(dataset)/1024

13.6781005859375

In [27]:
X = np.zeros((total_samples, 1024))
y = []
idx = 0

for key in dataset:
    num_samples = dataset[key].shape[0]
    X[idx : idx + num_samples, :] = dataset[key].cpu().numpy()
    # y[idx : idx + num_samples] = key
    if key in ["RightHand", "LeftHand"]:
        key = "Hand"
    y += [key] * num_samples
    idx += num_samples


In [23]:
key

'RightHand'

In [11]:
# using unsupervised learning, visualize clusters of data points


# Use T-SNE to reduce dimensionality of feature vectors to 2
from sklearn.manifold import TSNE


X_embedded = TSNE(n_components=2, ).fit_transform(X)



ValueError: Data must be 1-dimensional, got ndarray of shape (224102, 1) instead

In [25]:
import plotly.io as pio
pio.renderers.default = "browser"
px.scatter(x=X_embedded[:, 0], y=X_embedded[:, 1], color=y)

In [22]:
y[0]

0

In [28]:
# train a classifier to predict object from feature vector

import torch

# 3 layer neural network with 1024 input, 100 hidden, and 10 output, with batch normalization and leaky relu
class ObjectClassifier(torch.nn.Module):
    def __init__(self):
        super(ObjectClassifier, self).__init__()
        self.fc1 = torch.nn.Linear(1024, 512)
        self.fc2 = torch.nn.Linear(512, 128)
        self.fc3 = torch.nn.Linear(128, 16)
        self.bn1 = torch.nn.BatchNorm1d(512)
        self.bn2 = torch.nn.BatchNorm1d(128)
        self.relu = torch.nn.LeakyReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x


# train the classifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

model = ObjectClassifier()
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

from torchmetrics.classification import (MulticlassConfusionMatrix,
                                         MulticlassF1Score)

confusion_matrix = MulticlassConfusionMatrix(num_classes=16)
f1 = MulticlassF1Score(num_classes=16)


for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch} loss: {loss.item()}")
